# AI-Based Disease Prediction System — EDA & Model Training Notebook

- **Student Name:** Shaik Asif
- **Hall Ticket Number:** 2403A52220
- **Subject:** Artificial Intelligence & Deep Learning (AIDL)
- **Dataset:** Pima Indians Diabetes Database (NIDDK / UCI)

This notebook demonstrates exploratory data analysis, physiologically invalid zero detection, leakage-free scikit-learn pipelines, 5-fold stratified cross-validation, hold-out test evaluation, and sensitivity-prioritized model selection.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from backend.ml.preprocessing import FEATURE_ORDER, ZERO_AS_MISSING_COLUMNS, build_model_pipeline
from backend.ml.train import RANDOM_STATE, TEST_SIZE, CV_FOLDS, get_candidate_models

sns.set_theme(style='whitegrid')
df = pd.read_csv(PROJECT_ROOT / 'dataset' / 'pima_indians_diabetes.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## 1. Exploratory Data Analysis & Invalid Zero Detection
In the Pima Indians Diabetes Database, explicit `NaN` counts are 0, yet physiologically impossible `0` values appear in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`.

In [ ]:
print('Explicit NaN values per column:')
print(df.isna().sum())

print('\nClass distribution:')
print(df['Outcome'].value_counts().rename({0: 'Non-Diabetic (0)', 1: 'Diabetic (1)'}))

print('\nPhysiologically invalid zero counts:')
zero_summary = pd.DataFrame({
    'Invalid_Zeros': [(df[c] == 0).sum() for c in ZERO_AS_MISSING_COLUMNS],
    'Percent': [round((df[c] == 0).mean() * 100, 2) for c in ZERO_AS_MISSING_COLUMNS]
}, index=ZERO_AS_MISSING_COLUMNS)
zero_summary

## 2. Stratified Train/Test Split & Leakage-Free Pipeline Training
We split 80% training / 20% hold-out test **before** computing any imputation medians or feature scaling parameters so no test-set statistics leak into training.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

X = df[FEATURE_ORDER].copy()
y = df['Outcome'].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f'Train samples: {len(X_train)} (pos={y_train.sum()}) | Test samples: {len(X_test)} (pos={y_test.sum()})')

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

rows = []
for key, spec in get_candidate_models().items():
    pipe = build_model_pipeline(spec['estimator'], scale_features=spec['scale_features'])
    cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    cv_rec = cv_res['test_recall'].mean()
    cv_auc = cv_res['test_roc_auc'].mean()
    cv_f1 = cv_res['test_f1'].mean()
    clinical_score = 0.45 * cv_rec + 0.35 * cv_auc + 0.20 * cv_f1

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

    rows.append({
        'Model': spec['display_name'],
        'CV_Clinical_Score': round(clinical_score, 4),
        'Test_Accuracy': round(accuracy_score(y_test, preds), 4),
        'Test_Precision': round(precision_score(y_test, preds), 4),
        'Test_Recall': round(recall_score(y_test, preds), 4),
        'Test_F1': round(f1_score(y_test, preds), 4),
        'Test_ROC_AUC': round(roc_auc_score(y_test, probs), 4),
        'False_Negatives': int(fn)
    })

results_df = pd.DataFrame(rows).sort_values('CV_Clinical_Score', ascending=False)
results_df

## 3. Production Pipeline Export
Run `python backend/ml/train.py` from the repository root to regenerate `models/diabetes_pipeline.joblib`, `models/model_metadata.json`, `models/evaluation_metrics.json`, and all figures in `reports/figures/`.